# 📓 01 — 加载数据 + 基本探索

**目标**: 学会用 `load_prices()` 加载 parquet cache,做基本的数据探索。

**适合**: 第一次用 us-stock-causal 的用户。

---

**前置条件**: 已经跑过 `python examples/fetch_all.py`,data/raw/ 里有 parquet 文件。

如果还没跑,先开一个 terminal:
```bash
python examples/fetch_all.py
```


In [ ]:
# 第 1 步: import (含 robust path 修复 — 找含 src/ 的目录)
import sys
from pathlib import Path

def _find_project_root():
    cwd = Path.cwd()
    for cand in [cwd, *cwd.parents]:
        if (cand / 'src').is_dir() and (cand / 'config').is_dir():
            return cand
    return cwd

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

import pandas as pd
import matplotlib.pyplot as plt

from src.thresholds import load_prices

In [ ]:
# 第 2 步: 加载 4 指数的 1 年数据
tickers = ['DIA', 'QQQ', 'RSP', 'QQQE']
dfs = {sym: load_prices(sym, 'indices') for sym in tickers}

for sym, df in dfs.items():
    print(f"{sym:6s}: {len(df):>4d} rows, "
          f"{df.index[0].date()} → {df.index[-1].date()}, "
          f"close ${df['close'].iloc[-1]:.2f}")

In [ ]:
# 第 3 步: 算 1 日简单收益率 + 累计 5 日收益率
for sym, df in dfs.items():
    rets = df['close'].pct_change()
    cum_5d = (df['close'].iloc[-1] / df['close'].iloc[-6] - 1) * 100
    print(f"{sym:6s}: 1d {rets.iloc[-1]*100:+.2f}% | "
          f"5d cum {cum_5d:+.2f}%")

In [ ]:
# 第 4 步: 画 4 指数归一化对比 (起点 = 1.0)
fig, ax = plt.subplots(figsize=(12, 6))
for sym, df in dfs.items():
    norm = df['close'] / df['close'].iloc[0]
    ax.plot(df.index, norm, label=sym, linewidth=1.5)
ax.set_title("4 指数归一化对比 (起点 = 1.0)")
ax.set_ylabel("Normalized Close")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🎯 练习

试着自己改代码:

1. **改 lookback 周期**: 把 5 日累计改成 20 日累计
2. **加更多 ticker**: 加 `XLK` (科技) 和 `XLE` (能源),看哪些板块涨得多
3. **改 layer**: 把 `'indices'` 改成 `'sectors'`,看 11 GICS 行业
4. **加 volume 图**: `ax2 = ax.twinx(); ax2.bar(df.index, df['volume'], alpha=0.3)`

改完跑一遍,看输出怎么变。**这就是 self-analysis 的核心**: 改参数 → 看结果。